# E3 — XBART (GFR)

[Artículo](https://doi.org/10.1080/01621459.2021.1942012). Experimento solicitado de la búsqueda Scopus.

Se usa `stochtree==0.4.5`, `BARTModel` con Grow-From-Root puro:
60 barridos GFR, cero MCMC y descarte de los primeros20 al promediar las40
predicciones restantes. Es la arquitectura XBART implementada en stochtree,
no una ejecución del binario original de JingyuHe. Se ajusta la varianza de
las hojas y se fija semilla42. Árboles50/150 × profundidad2/4 × hoja10/30.

Preparación canónica SVM dentro del fold, con one-hot. stochtree advierte
limitaciones del muestreo GFR en predictores discretos/repetidos; se conserva
esta representación para reutilizar el tratamiento canónico y se documenta
la limitación. No se añaden iteraciones MCMC porque sería otro estimador.
La media predictiva regresa en la escala original del objetivo.

## Protocolo compartido

Misma fuente real `Data/Datos_Brutos.csv`, reglas de negocio y15 predictores.
Objetivo `nota_primer_parcial`; nota_final, pidm, periodo e índice de fila no
son entradas. Sin aumentación ni eliminación de outliers válidos.
Train1309 filas/1293 estudiantes; test202530:188 estudiantes, sin solapamiento.

Cinco folds externos canónicos por estudiante, cinco internos en cada búsqueda
y cinco internos para elegir parámetros finales. RMSE medio interno selecciona
la configuración. Cada preprocesador y modelo se ajusta solo en su fit.
La selección final se guarda antes de predecir test. Mismas funciones del tablero:
RMSE, MAE, MAPE(%) y R²; métricas sin ponderación, escala original.
Media/SD entre folds, OOF reunido y test temporal se informan por separado.

Se guardan grillas, auditorías, hashes, OOF y predicciones test. El código puede
reanudar cálculos de igual procedencia; si cambia la procedencia exige una
carpeta nueva. Pesos/estimadores quedan locales en modelos_preentrenados/scopus.
No se reemplazan los modelos del dashboard ni se reescriben informes anteriores.

In [1]:
from pathlib import Path
import sys
BOOK=Path.cwd() if (Path.cwd()/'entregable3.py').exists() else Path.cwd()/'Precalculo'
sys.path.insert(0,str(BOOK.resolve()))
import pandas as pd
from IPython.display import display
from evaluar_scopus_e3 import run,compare,ROOT

report=run('xbart')
display(pd.DataFrame(report['cv']).round(6))
print('Parámetros finales:',report['best_params'])

,metrica,media_cv,sd_cv,OOF_agrupado
0,RMSE,0.882018,0.033427,0.882626
1,MAE,0.712260,0.025582,0.712248
2,MAPE (%),26.858646,1.990018,26.857500
3,R²,0.375379,0.027339,0.376296


Parámetros finales: {'depth': 4, 'leaf': 10, 'trees': 50}


## Test temporal y auditoría

El test no selecciona hiperparámetros.

In [2]:
display(pd.DataFrame([report['test']]).round(6))
display(pd.DataFrame(report['fold_audit']))

,RMSE,MAE,MAPE (%),R²
0,0.967515,0.794435,31.849667,0.378447


,fold,best_params,inner_rmse,fit_rows,validation_rows,shared_students,seconds
0,1,"{'depth': 4, 'leaf': 30, 'trees': 50}",0.887375,1047,262,0,218.406764
1,2,"{'depth': 4, 'leaf': 10, 'trees': 50}",0.866641,1047,262,0,42.273824
2,3,"{'depth': 4, 'leaf': 10, 'trees': 50}",0.862138,1047,262,0,258.830062
3,4,"{'depth': 4, 'leaf': 10, 'trees': 50}",0.868453,1047,262,0,257.467372
4,5,"{'depth': 4, 'leaf': 10, 'trees': 50}",0.871261,1048,261,0,260.829601


## Comparación conjunta

Tras completar los cuatro modelos, consultar precalculo_scopus_e3.ipynb y resultados/e3/scopus/comparacion.json.